# 07 — Retrieval

**Lernziel.** Du lädst Dokumente, zerlegst sie, machst sie durchsuchbar und hängst
die Suche als Werkzeug in den Agenten. Findet die Suche nichts, springt die
Websuche aus Woche 6 ein.

**Dauer.** Rund 35 Minuten.

**Voraussetzungen.** Notebook 06 gelaufen. Die Dateien in `data/`.

**Empfohlen:** lokal ausführen, zum Beispiel in PyCharm. Das Embedding-Modell lädt
beim ersten Mal rund 90 MB herunter.

---

Wieder am selben Fall: ein Prototyp, der Fragen zu einer Reise mit SWISS International Air Lines beantwortet.

## Setup

Dieselben Zellen wie in Woche 6.

In [46]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [47]:
LLM_MODEL = "gpt-5.6-luna"
LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"

# Weitere Modelle auf demselben Proxy:
#   "azure_ai/mistral-large-3"

In [48]:
import os
from dotenv import load_dotenv
from getpass import getpass

# load_dotenv liest die .env, überschreibt aber nichts, was schon gesetzt ist.
load_dotenv()

if not os.environ.get("LITELLM_API_KEY"):
    os.environ["LITELLM_API_KEY"] = getpass("LITELLM_API_KEY: ")

In [49]:
# vor dem ersten Modellaufruf setzen, sonst greift das Tracing nicht
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "genai_swiss_cw7"
    print("Tracing läuft. Projekt:", os.environ["LANGSMITH_PROJECT"])
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("Kein LANGSMITH_API_KEY gefunden — weiter ohne Tracing.")

Tracing läuft. Projekt: genai_swiss_cw7


In [50]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=os.environ["LITELLM_API_KEY"],
    temperature=0,
)

print(llm.invoke("Antworte mit einem Wort: bereit?").content)

Bereit.


## Stand aus Woche 6

Der Graph aus Woche 5 mit den Werkzeugen aus Woche 6: `fare_lookup`, Tavily (falls
Schlüssel vorhanden), `ToolNode` mit Fehlerbehandlung und `RetryPolicy`.

```
START → classify → reject → END
START → classify → answer ⇄ tools → respond → END
```

Ohne MCP, damit alles synchron bleibt. Wer MCP-Werkzeuge dazunimmt, macht wie in
Woche 6 `call_tools` asynchron und ruft `await graph.ainvoke(...)` auf.

In [51]:
import operator, os
from typing import Literal
from typing_extensions import Annotated, TypedDict
from pydantic import BaseModel, Field
from langchain.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import RetryPolicy

SYSTEM_PROMPT = """Du beantwortest Fragen zu Tarifen und Gepäck bei SWISS.
Fakten: Schlage Konditionen und Gebühren mit fare_lookup nach. Erfinde nichts, auch keine Kontaktdaten.
Fehlt eine Angabe, frage nach.
Antwortstil: Sprich die Person mit «Sie» an, antworte in der Sprache der Frage, höchstens vier Sätze.
Kennzeichnung: Du bist ein Prototyp aus einer Lehrveranstaltung, kein offizieller Dienst von SWISS.
Websuche: Was nicht in den Tarifdaten steht, etwa Flugverbindungen, suchst du mit tavily_search.
Nenne dann die Quelle (Adresse) und sage, dass es eine Information aus dem Internet ist."""

CLASSIFY_PROMPT = """Entscheide, ob eine Frage zu einem Assistenten für SWISS International Air Lines gehört, und bestimme ihr Thema.
Zuständig: Flüge, Tarife, Gepäck, Buchung, Umbuchung, Erstattung bei SWISS, Fragen an den Assistenten.
Nicht zuständig: andere Fluggesellschaften und alles andere."""

FARES = {
    "economy light":   {"carry_on": "1 Stück, 8 kg", "checked_baggage": "kein Freigepäck",
                        "extra_bag_fee": "CHF 60", "rebooking_fee": "nicht umbuchbar",
                        "seat_selection": "ab CHF 15"},
    "economy classic": {"carry_on": "1 Stück, 8 kg", "checked_baggage": "1 Stück, 23 kg",
                        "extra_bag_fee": "CHF 60", "rebooking_fee": "CHF 80",
                        "seat_selection": "ab CHF 15"},
    "economy flex":    {"carry_on": "1 Stück, 8 kg", "checked_baggage": "1 Stück, 23 kg",
                        "extra_bag_fee": "CHF 60", "rebooking_fee": "kostenlos",
                        "seat_selection": "inklusive"},
}
BAGGAGE_KEYS = ["carry_on", "checked_baggage", "extra_bag_fee"]


@tool
def fare_lookup(fare: str, runtime: ToolRuntime) -> str:
    """Schlägt Konditionen und Nebenkosten eines SWISS-Tarifs nach.

    Liefert Handgepäck, Freigepäck, Gebühr für ein zusätzliches Gepäckstück,
    Umbuchungsgebühr und Sitzplatzwahl.
    Mögliche Tarife: Economy Light, Economy Classic, Economy Flex.
    """
    data = FARES.get(fare.strip().lower())
    if data is None:
        return f"Tarif '{fare}' ist nicht in der Tabelle. Bekannt sind: {', '.join(FARES)}."
    if runtime.state["category"] == "baggage":
        data = {k: v for k, v in data.items() if k in BAGGAGE_KEYS}
    return ", ".join(f"{k}: {v}" for k, v in data.items())


tools = [fare_lookup]
if os.environ.get("TAVILY_API_KEY"):
    web_search = TavilySearch(max_results=3)
    tools.append(web_search)
llm_with_tools = llm.bind_tools(tools)


class Classification(BaseModel):
    reason: str = Field(description="Begründung in einem Satz")
    in_scope: bool = Field(description="True, wenn die Frage in den Zuständigkeitsbereich fällt")
    category: Literal["fare", "baggage", "booking", "refund", "other"] = Field(description="Thema der Frage")


class AgentResponse(BaseModel):
    answer: str = Field(description="Antwort an die Kundin oder den Kunden")
    fare: str | None = Field(description="Tarif, um den es geht, falls einer genannt wird")
    needs_human: bool = Field(description="True bei Beschwerden, Entschädigungsforderungen, Notlagen unterwegs "
                                          "oder wenn eine Buchung geändert werden muss")


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    in_scope: bool
    category: str
    tools_used: Annotated[list[str], operator.add]
    response: AgentResponse


classifier = llm.with_structured_output(Classification, method="json_schema")
responder = llm.with_structured_output(AgentResponse, method="json_schema")


def handle_tool_errors(request, handler):
    try:
        return handler(request)
    except ConnectionError:
        raise
    except Exception as e:
        return ToolMessage(content=f"Fehler in {request.tool_call['name']}: {e}. "
                                   "Sag der Person, was nicht geklappt hat.",
                           tool_call_id=request.tool_call["id"])


tool_node = ToolNode(tools, wrap_tool_call=handle_tool_errors)


def classify(state: AgentState):
    result = classifier.invoke([SystemMessage(content=CLASSIFY_PROMPT), state["messages"][-1]])
    return {"in_scope": result.in_scope, "category": result.category}


def reject(state: AgentState):
    text = "Dabei kann ich leider nicht helfen. Ich beantworte nur Fragen zu SWISS."
    return {"messages": [AIMessage(content=text)],
            "response": AgentResponse(answer=text, fare=None, needs_human=False)}


def answer(state: AgentState):
    return {"messages": [llm_with_tools.invoke([SystemMessage(content=SYSTEM_PROMPT)] + state["messages"])]}


def call_tools(state: AgentState):
    result = tool_node.invoke(state)
    return {"messages": result["messages"],
            "tools_used": [c["name"] for c in state["messages"][-1].tool_calls]}


def respond(state: AgentState):
    messages = ([SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
                + [HumanMessage(content="Übernimm deine letzte Antwort unverändert in das vorgegebene Format.")])
    return {"response": responder.invoke(messages)}


def route_after_classify(state: AgentState):
    return "answer" if state["in_scope"] else "reject"


def route_after_answer(state: AgentState):
    return "tools" if state["messages"][-1].tool_calls else "respond"


def build_graph():
    builder = StateGraph(AgentState)
    builder.add_node("classify", classify)
    builder.add_node("reject", reject)
    builder.add_node("answer", answer)
    builder.add_node("tools", call_tools,
                     retry_policy=RetryPolicy(max_attempts=3, retry_on=ConnectionError))
    builder.add_node("respond", respond)
    builder.add_edge(START, "classify")
    builder.add_conditional_edges("classify", route_after_classify, ["answer", "reject"])
    builder.add_edge("reject", END)
    builder.add_conditional_edges("answer", route_after_answer, ["tools", "respond"])
    builder.add_edge("tools", "answer")
    builder.add_edge("respond", END)
    return builder.compile()


graph = build_graph()


def ask(question):
    return graph.invoke({"messages": [HumanMessage(content=question)]})["response"].answer


print("Werkzeuge:", [t.name for t in tools])

Werkzeuge: ['fare_lookup', 'tavily_search']


Heute kommt ein Werkzeug dazu: die Suche in Dokumenten. Bis es so weit ist, sind
es fünf Schritte: laden, aufbereiten, zerlegen, einbetten, speichern.

## 1. Laden

Die Dateien liegen in `data/` und werden nie von der Website geholt: SWISS erlaubt
kein automatisches Auslesen, bei anderen Fällen könnte man die aktuellsten Dokumente per Webscraping holen.

Ein PDF und fünf gespeicherte Webseiten, je mit dem passenden Loader. Beide stammen
aus `langchain-community`. Das Paket wird nicht mehr weiterentwickelt, daher die
Warnung; die Loader funktionieren aber.

In [52]:
import glob, logging
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader

logging.getLogger("pypdf").setLevel(logging.ERROR)  # harmlose Meldungen zum PDF ausblenden

pdf_docs = PyPDFLoader("data/General Conditions of Carriage | SWISS.pdf").load()
html_docs = [BSHTMLLoader(path, open_encoding="utf-8").load()[0]
             for path in sorted(glob.glob("data/*.html"))]

print(len(pdf_docs), "PDF-Seiten,", len(html_docs), "Webseiten")
print(repr(pdf_docs[34].page_content[:300]))

37 PDF-Seiten, 5 Webseiten
'20.10.26, 21:27General Conditions of Carriage | SWISS\nSeite 35 von 37https://www.swiss.com/ch/en/terms-conditions/conditions-of-carriage\n\uf801\uf806.\uf805 Liability in the case of\ncodeshare flights\nIn the case of flights operated by us as codeshare services in\nassociation with another carrier, we will be liable '


**Zum Vergleich: dasselbe ohne Loader.** Ein Loader liest die Datei und packt den
Text in ein `Document`, mehr nicht. Mit `pypdf` und `BeautifulSoup` direkt kommt
derselbe Text heraus. Wir nehmen als Beispiel nun die community loaders, aber grundsätzlich ist es egal, wie die Dokumente geladen werden.

In [53]:
from pypdf import PdfReader
from bs4 import BeautifulSoup
from langchain_core.documents import Document

path = "data/General Conditions of Carriage | SWISS.pdf"
pdf_docs_plain = [Document(page_content=page.extract_text(), metadata={"source": path, "page": i})
                  for i, page in enumerate(PdfReader(path).pages)]

html_docs_plain = []
for path in sorted(glob.glob("data/*.html")):
    with open(path, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")
    html_docs_plain.append(Document(page_content=soup.get_text(), metadata={"source": path}))

print("PDF gleich:", [d.page_content for d in pdf_docs_plain] == [d.page_content for d in pdf_docs])
print("HTML gleich:", [d.page_content for d in html_docs_plain] == [d.page_content for d in html_docs])

PDF gleich: True
HTML gleich: True


In [54]:
print(pdf_docs[0].page_content[:300])

20.10.26, 21:27General Conditions of Carriage | SWISS
Seite 1 von 37https://www.swiss.com/ch/en/terms-conditions/conditions-of-carriage
General Conditions of
Carriage
Last updated: January 
Terms and Conditons of Carriage in Romanian
As you read these Conditions, please note the following:
"We",


## 2. Aufbereiten / Preprocessing

So sieht der Text aus, den die Suche zu sehen bekommt:

- Im PDF fehlen die Ziffern: Der Browser hat sie beim Drucken als Sonderzeichen
  abgelegt (`\uf807` statt `7`). Ausserdem beginnt jede Seite mit Datum und Adresse.
- In den Webseiten steht vor dem Inhalt die ganze Navigation.

Ohne Aufbereitung landet das alles im Index.

In [55]:
import re
from langchain_core.documents import Document

DIGITS = {0xF800 + i: str(i) for i in range(10)}  # Sonderzeichen → Ziffer

# neue Listen statt Änderungen an pdf_docs / html_docs: so bleibt das Original erhalten
pdf_clean = []
for doc in pdf_docs:
    lines = doc.page_content.translate(DIGITS).split("\n")
    text = "\n".join(lines[2:])  # Kopfzeilen mit Datum und Adresse weg
    pdf_clean.append(Document(page_content=text, metadata=doc.metadata))

html_clean = []
for doc in html_docs:
    text = doc.page_content.split("Main content", 1)[-1]  # Navigation weg
    text = re.sub(r"\s*\n\s*", "\n", text).strip()  # leere Zeilen weg
    html_clean.append(Document(page_content=text, metadata=doc.metadata))

docs = pdf_clean + html_clean
print(repr(pdf_clean[34].page_content[:300]))

'16.5 Liability in the case of\ncodeshare flights\nIn the case of flights operated by us as codeshare services in\nassociation with another carrier, we will be liable to those\npassengers who concluded a contract of carriage with us and on\nwhose Tickets we are shown as the carrier by our designation code'


In [56]:
print(pdf_clean[0].page_content[:300])

General Conditions of
Carriage
Last updated: January 2025
Terms and Conditons of Carriage in Romanian
As you read these Conditions, please note the following:
"We", "our", "ourselves" and "us" means Swiss International Air
Lines Ltd., domiciled in Kloten, Switzerland.
"You", "your" and "yourself" me


## 3. Splitting

37 Seiten passen nicht sinnvoll in einen Prompt. `RecursiveCharacterTextSplitter`
zerlegt in Stücke (Chunks) und trennt bevorzugt an Absätzen, dann an Zeilen.
`chunk_overlap` sorgt dafür, dass ein Satz auf der Grenze in beiden Nachbarn steht.

In [57]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print(len(docs), "Dokumente, insgesamt vor dem Splitten")
print("\n++++++++++++++++++++++++++++\n")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)

print(len(chunks), "Chunks")
print(chunks[0].metadata["source"], chunks[0].page_content[:200])

42 Dokumente, insgesamt vor dem Splitten

++++++++++++++++++++++++++++

137 Chunks
data/General Conditions of Carriage | SWISS.pdf General Conditions of
Carriage
Last updated: January 2025
Terms and Conditons of Carriage in Romanian
As you read these Conditions, please note the following:
"We", "our", "ourselves" and "us" means S


## 4. Embeddings

Ein Embedding macht aus Text einen Vektor. Ähnliche Vektoren liegen nah beieinander. Das Modell läuft lokal und kostet nichts. Es versteht nur
Englisch — die Dokumente auch, also suchen wir englisch.

In [58]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings1 = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vector = embeddings1.embed_query("My flight was cancelled.")
print(len(vector), "Zahlen, die ersten fünf:", [round(x, 3) for x in vector[:5]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384 Zahlen, die ersten fünf: [0.079, 0.01, 0.043, 0.065, 0.019]


In [59]:
# Mehrsprachig über den LiteLLM-Proxy

from langchain_openai import OpenAIEmbeddings
embeddings2 = OpenAIEmbeddings(
    model="andreasmartin/apertus-v1.1-swiss-embed-0.4b",
    base_url=LLM_BASE_URL,
    api_key=os.environ["LITELLM_API_KEY"],
    check_embedding_ctx_length=False,        # nicht mit dem OpenAI-Tokenizer vorzerlegen
    model_kwargs={"encoding_format": None},  # der Proxy lehnt diesen Parameter ab
)

vector = embeddings2.embed_query("My flight was cancelled.")
print(len(vector), "Zahlen, die ersten fünf:", [round(x, 3) for x in vector[:5]])

1024 Zahlen, die ersten fünf: [-0.022, 0.018, -0.023, -0.013, 0.01]


## 5. Die Vektordatenbank: Chroma

Chroma speichert die Chunks mit ihren Vektoren in einer **Collection**. Ohne
Angabe eines Ordners bleibt alles im Arbeitsspeicher; mit
`persist_directory="chroma_db"` würde es auf die Festplatte geschrieben.
`"cosine"` liefert Ähnlichkeiten zwischen 0 und 1.

In [60]:
from langchain_chroma import Chroma

store = Chroma(collection_name="swiss_docs", embedding_function=embeddings1,
               collection_metadata={"hnsw:space": "cosine"})
store.reset_collection()  # sonst kommt beim zweiten Ausführen alles doppelt hinein
store.add_documents(chunks)

QUESTION = "How long do I have to report damaged baggage?"

for doc, score in store.similarity_search_with_relevance_scores(QUESTION, k=3):
    print(f"Score: {score:.2f}", "\nMetadata: ", doc.metadata["source"][5:], "|", doc.page_content[:120].replace("\n", " "))

Score: 0.72 
Metadata:  General Conditions of Carriage | SWISS.pdf | checked baggage, you must notify us as soon as you discover the damage and within seven (7) days at the latest of receip
Score: 0.55 
Metadata:  General Conditions of Carriage | SWISS.pdf | 16.5 Liability in the case of codeshare flights In the case of flights operated by us as codeshare services in associati
Score: 0.54 
Metadata:  General Conditions of Carriage | SWISS.pdf | the damage resulted from the inherent defect, quality or vice of the baggage. In the case of unchecked baggage, includin


## 6. Zurück zum Splitting: Wie gross soll ein Chunk sein?

Dieselbe Frage, drei Zerlegungen, jeweils der beste Treffer.

In [61]:
for size in [200, 1000, 3000]:
    parts = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 8).split_documents(docs)
    test_store = Chroma(collection_name=f"chunks_{size}", embedding_function=embeddings,
                        collection_metadata={"hnsw:space": "cosine"})
    test_store.reset_collection()
    test_store.add_documents(parts)
    best = test_store.similarity_search(QUESTION, k=1)[0]
    print(f"=== chunk_size={size}: {len(parts)} Chunks, Treffer mit {len(best.page_content)} Zeichen ===")
    print(best.page_content[:400], "\n")

=== chunk_size=200: 654 Chunks, Treffer mit 72 Zeichen ===
Further information on dangerous items
Stowing carry-on baggage on board 

=== chunk_size=1000: 130 Chunks, Treffer mit 560 Zeichen ===
List of prohibited and dangerous items
Frequently asked questions about carry-on baggage
When travelling with multiple airlines, the carry-on baggage rules of the operating carrier apply to each individual flight. This includes flights operated by another airline under a SWISS flight number (e.g. LX123) – the carry-on baggage rules of the operating carrier apply in these cases too. Before you trav 

=== chunk_size=3000: 54 Chunks, Treffer mit 2992 Zeichen ===
Foldable pushchair
1 per infant (pocket buggy)
Duty-free purchases
Your duty-free purchase does not count as an additional baggage item and must not cause your carry-on allowance to be exceeded.
Small musical instruments
(max. 55 × 40 × 23 cm und 8 kg)
Special rules apply to the transport of musical instruments.
Information about the trans

Klein ist präzise, aber der Satz bricht ab. Gross bringt Zusammenhang, aber auch
viel Text, der mit der Frage nichts zu tun hat. Einen richtigen Wert gibt es nicht,
nur einen passenden zu den eigenen Dokumenten.

## 7. Dense und sparse

Embeddings suchen nach **Bedeutung** (dense). BM25 zählt **Wörter** (sparse): kein
Modell, keine Vektoren. Zwei Anfragen, je der beste Treffer:

In [62]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(chunks, k=1)

for query in [QUESTION, "Regulation 261/2004"]:
    print(f"=== {query} ===")
    print("sparse:", bm25.invoke(query)[0].page_content[:100].replace("\n", " "))
    print("dense: ", store.similarity_search(query, k=1)[0].page_content[:100].replace("\n", " "), "\n")

=== How long do I have to report damaged baggage? ===
sparse: Request ticket refund manually Please check and complete your details before submitting the form. I 
dense:  checked baggage, you must notify us as soon as you discover the damage and within seven (7) days at  

=== Regulation 261/2004 ===
sparse: EU Regulation 261/04: your rights as a passenger You are entitled to certain rights under the terms 
dense:  stopped. The method of calculating the period of limitation shall be determined according to the law 



Bei der Frage in eigenen Worten gewinnt dense, beim genauen Begriff sparse. In der
Praxis kombiniert man oft beides («hybrid»).

Wie gross ist ein sparse Vektor? Jedes Wort ist eine Dimension, ein Chunk belegt nur
die Wörter, die in ihm vorkommen. BM25 trennt dabei nur an Leerzeichen: `baggage?` und
`baggage` sind zwei verschiedene Wörter.

In [63]:
vocabulary = bm25.vectorizer.idf          # jedes Wort, das in irgendeinem Chunk vorkommt
chunk = bm25.vectorizer.doc_freqs[0]       # Wort → Häufigkeit im ersten Chunk

print(len(vocabulary), "Dimensionen sparse (ein Wort = eine Dimension)")
print(len(chunk), "davon im ersten Chunk nicht null")
print(sorted(chunk.items(), key=lambda x: -x[1])[:8])
print(sorted(QUESTION.split()), "→ Die Frage belegt", len(set(QUESTION.split())), "Dimensionen")
print(len(vector), "Dimensionen dense, alle nicht null")

3265 Dimensionen sparse (ein Wort = eine Dimension)
98 davon im ersten Chunk nicht null
[('the', 7), ('of', 6), ('in', 5), ('Carriage', 3), ('and', 3), ('means', 3), ('to', 3), ('Convention', 3)]
['How', 'I', 'baggage?', 'damaged', 'do', 'have', 'long', 'report', 'to'] → Die Frage belegt 9 Dimensionen
1024 Dimensionen dense, alle nicht null


**Hybrid: beide Listen zusammenführen (RRF).** Zuerst trennen wir besser: Kleinbuchstaben,
nur Wörter und Zahlen. Dann bekommt jeder Chunk pro Liste `1 / (60 + Rang)` Punkte, die
Punkte werden addiert. Wer in beiden Listen weit oben steht, gewinnt. LangChain hat
dafür auch eine fertige Klasse, `EnsembleRetriever` in `langchain_classic.retrievers`.

In [64]:
import re

def words(text):
    return re.findall(r"\w+", text.lower())  # Kleinbuchstaben, nur Wörter und Zahlen

bm25_words = BM25Retriever.from_documents(chunks, k=10, preprocess_func=words)

def hybrid_search(query, k=3):
    sparse = bm25_words.invoke(query)
    dense = store.similarity_search(query, k=10)
    scores = {}
    for results in [sparse, dense]:
        for rank, doc in enumerate(results, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (60 + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

for query in [QUESTION, "Regulation 261/2004"]:
    print(f"=== {query} ===")
    for text, score in hybrid_search(query):
        print(f"{score:.4f}", text[:90].replace("\n", " "))

=== How long do I have to report damaged baggage? ===
0.0164 5.5.1 Onward or return reservations may have to be reconfirmed within specified time limit
0.0164 checked baggage, you must notify us as soon as you discover the damage and within seven (7
0.0161 Please make sure that you pack your sports equipment carefully to ensure that it is not da
=== Regulation 261/2004 ===
0.0325 EU Regulation 261/04: your rights as a passenger You are entitled to certain rights under 
0.0318 stopped. The method of calculating the period of limitation shall be determined according 
0.0302 Please contact our ​Customer Feedback Services Department to check your entitlements under


0,0164 = 1/61: nur in einer Liste auf Platz 1. 0,0325: in beiden Listen vorne. Bei der
Frage zum Koffer findet BM25 den richtigen Chunk gar nicht («report damaged» gegen
«notify … damage»), also gibt es einen Gleichstand mit einem falschen Treffer. RRF
belohnt Übereinstimmung, es weiss nicht, welche Liste recht hat.

## 8. Die Suche als Werkzeug

Ein Retriever mit Schwellwert: Chunks unter `SIMILARITY_THRESHOLD` werden
weggelassen. Bleibt nichts übrig, sagt das Werkzeug das ausdrücklich, statt
schwache Treffer zu liefern. Der Wert ist ein Startpunkt — probier andere aus.

In [65]:
SIMILARITY_THRESHOLD = 0.5

retriever = store.as_retriever(search_type="similarity_score_threshold",
                               search_kwargs={"k": 4, "score_threshold": SIMILARITY_THRESHOLD})


@tool
def search_documents(query: str) -> str:
    """Durchsucht die Unterlagen von SWISS: Beförderungsbedingungen, Rückerstattung,
    Passagierrechte (EU 261), Handgepäck und Sportgepäck.
    Die Unterlagen sind englisch, formuliere die Suchanfrage auf Englisch.
    """
    hits = retriever.invoke(query)
    if not hits:
        return "In den Dokumenten nichts gefunden."
    return "\n\n".join(f"[{doc.metadata['source'][5:]}]\n{doc.page_content}" for doc in hits)


print(search_documents.invoke({"query": QUESTION})[:300], "\n")
print(search_documents.invoke({"query": "SWISS direct flights Zurich Bangkok"}))

[General Conditions of Carriage | SWISS.pdf]
checked baggage, you must notify us as soon as you discover the
damage and within seven (7) days at the latest of receipt of the
baggage. If you wish to file a claim or an action regarding a delay to
the delivery of Checked Baggage, you must notify us wit 



No relevant docs were retrieved using the relevance score threshold 0.5


In den Dokumenten nichts gefunden.


Der Docstring verlangt eine englische Suchanfrage. So kann man den Agenten auf
Deutsch fragen: Das Modell übersetzt selbst, bevor es sucht.

## 9. Ins Werkzeug-Set

Wie jedes Werkzeug: in die Liste `tools`, neu binden, neuer `ToolNode`. Dazu eine
Zeile im System-Prompt, die die Reihenfolge festlegt: zuerst die Dokumente, dann
das Internet. Der Graph bleibt derselbe.

In [66]:
SYSTEM_PROMPT = SYSTEM_PROMPT + """
Unterlagen: Regeln und Bedingungen (Erstattung, Passagierrechte, Gepäck) suchst du zuerst mit search_documents.
Geben die Unterlagen keine Antwort, suchst du mit tavily_search."""

tools = tools + [search_documents]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools, wrap_tool_call=handle_tool_errors)

print("Werkzeuge:", [t.name for t in tools])

Werkzeuge: ['fare_lookup', 'tavily_search', 'search_documents']


Zuerst das blanke Modell, dann der Agent. Dieselbe Frage, diesmal auf Deutsch.

In [67]:
DOC_QUESTION = "Mein Koffer ist beschädigt angekommen. Bis wann muss ich das melden?"

print(llm.invoke(DOC_QUESTION).content)

Beschädigtes **aufgegebenes Gepäck** sollten Sie:

- **sofort am Flughafen** beim Gepäckschalter melden und eine **PIR-/Schadensmeldung** aufnehmen lassen – möglichst bevor Sie den Flughafen verlassen.
- Zusätzlich der Airline **schriftlich spätestens innerhalb von 7 Tagen nach Erhalt des Koffers** melden (nach dem Montrealer Übereinkommen).

Bewahren Sie Gepäckabschnitt, Bordkarte und PIR-Nummer auf und machen Sie Fotos vom Schaden. Bei verspäteter Zustellung gilt meist eine Frist von **21 Tagen**. Bei einer Pauschalreise sollten Sie den Schaden außerdem unverzüglich dem Reiseveranstalter melden.


In [68]:
result = graph.invoke({"messages": [HumanMessage(content=DOC_QUESTION)]})
print("tools_used:", result["tools_used"])
print(result["response"])

tools_used: ['search_documents']
answer='Sie müssen den Schaden so bald wie möglich, spätestens jedoch **innerhalb von 7 Tagen nach Erhalt des aufgegebenen Gepäcks** schriftlich melden. Ich bin ein Prototyp aus einer Lehrveranstaltung und kein offizieller Dienst von SWISS.' fare=None needs_human=True


Die erste Antwort klingt gut und ist vielleicht sogar richtig — aber nicht
überprüfbar. Die zweite stützt sich auf die Beförderungsbedingungen; im Trace
(`genai_swiss_cw7`) siehst du, welche Chunks das Modell bekommen hat.

Und eine Frage, bei der die Dokumente nicht weiterhelfen:

In [69]:
result = graph.invoke({"messages": [HumanMessage(content="Darf ich meine Katze in der Kabine mitnehmen?")]})
print("tools_used:", result["tools_used"])
print(result["response"].answer)

tools_used: ['search_documents', 'search_documents']
Ja, Sie dürfen Ihre Katze grundsätzlich in der Kabine mitnehmen, sofern SWISS sie akzeptiert und sie samt Transporttasche in den Fussraum unter dem Vordersitz passt. Die Katze muss während des gesamten Flugs sicher transportiert werden und sich angemessen verhalten; sie zählt nicht zum Freigepäck und ist kostenpflichtig. Ob dies auf Ihrer konkreten Strecke möglich ist, hängt unter anderem von Zielland und Flug ab – bitte nennen Sie mir Strecke und Reisedatum. Ich bin ein Prototyp aus einer Lehrveranstaltung und kein offizieller Dienst von SWISS.


Zuerst die Dokumente, dann das Internet: `tools_used` zeigt die Reihenfolge. Ohne
`TAVILY_API_KEY` antwortet der Agent ohne Websuche — wie er damit umgehen soll, ist
ein Thema für die Guardrails.

## Aufgaben bis zur nächsten Woche:

- **Testfragen:** Formuliert 5 Fragen zu euren Dokumenten und nutzt sie für alle
  folgenden Vergleiche.
- **Eigene Dokumente:** Legt eure Unterlagen in `data/` und ladet sie. Was muss
  bereinigt werden?
- **Chunks:** Verändert `chunk_size` und `chunk_overlap`. Welche Treffer
  verbessern sich?
- **Schwellwert:** Ab welchem `SIMILARITY_THRESHOLD` kommen falsche Treffer durch,
  ab welchem fehlen richtige?
- **Embeddings:** Wechselt auf ein mehrsprachiges Modell (siehe
  [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)) und sucht
  direkt auf Deutsch. Wie ändern sich die Werte?
- **Retrieval:** Vergleicht dense, sparse und hybrides Retrieval (RRF).

### Für euer Projekt

- **Braucht euer Agent RAG?** Begründet eure Entscheidung in 2–3 Sätzen.

## Baustein zum Mitnehmen

In [70]:
import glob, re
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.prebuilt import ToolNode

pdf_docs = PyPDFLoader("data/General Conditions of Carriage | SWISS.pdf").load()
html_docs = [BSHTMLLoader(path, open_encoding="utf-8").load()[0]
             for path in sorted(glob.glob("data/*.html"))]

DIGITS = {0xF800 + i: str(i) for i in range(10)}
for doc in pdf_docs:
    doc.page_content = "\n".join(doc.page_content.translate(DIGITS).split("\n")[2:])
for doc in html_docs:
    doc.page_content = re.sub(r"\s*\n\s*", "\n", doc.page_content.split("Main content", 1)[-1]).strip()

chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(pdf_docs + html_docs)

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

store = Chroma(collection_name="swiss_docs", embedding_function=embeddings,
               collection_metadata={"hnsw:space": "cosine"})
store.reset_collection()
store.add_documents(chunks)

SIMILARITY_THRESHOLD = 0.5
retriever = store.as_retriever(search_type="similarity_score_threshold",
                               search_kwargs={"k": 4, "score_threshold": SIMILARITY_THRESHOLD})


@tool
def search_documents(query: str) -> str:
    """Durchsucht die Unterlagen von SWISS: Beförderungsbedingungen, Rückerstattung,
    Passagierrechte (EU 261), Handgepäck und Sportgepäck.
    Die Unterlagen sind englisch, formuliere die Suchanfrage auf Englisch.
    """
    hits = retriever.invoke(query)
    if not hits:
        return "In den Dokumenten nichts gefunden."
    return "\n\n".join(f"[{doc.metadata['source'][5:]}]\n{doc.page_content}" for doc in hits)


SYSTEM_PROMPT = SYSTEM_PROMPT + """
Unterlagen: Regeln und Bedingungen (Erstattung, Passagierrechte, Gepäck) suchst du zuerst mit search_documents.
Geben die Unterlagen keine Antwort, suchst du mit tavily_search."""

tools = [fare_lookup, search_documents]
if os.environ.get("TAVILY_API_KEY"):
    tools.append(TavilySearch(max_results=3))

llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools, wrap_tool_call=handle_tool_errors)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]